In [ ]:
# -*- coding: utf-8 -*-
"""
FULL river-temperature mapping workflow (Bode) — publication-ready
------------------------------------------------------------------
What this script does
1) Predicts river water temperature for ONE ECOSTRESS overpass
2) Optionally predicts many readable scenes and builds:
   - long-term annual mean map
   - seasonal mean maps (DJF, MAM, JJA, SON)
3) Produces publication-ready figures in:
   - projected CRS (EPSG:25832)  -> recommended main figures
   - WGS84 / lat-lon (EPSG:4326) -> optional supplementary/context figures

Robustness additions
- Prefers h5netcdf, then scipy, then netcdf4
- Opens ECOSTRESS stack with only LST_C (drops QC/CLOUD/WATER)
- Skips unreadable ECOSTRESS scenes instead of crashing
- Loads slices to memory before interpolation to reduce HDF errors
- Supports optional local copy of the ECOSTRESS stack

Recommended first test
- RUN_SINGLE_SCENE = True
- RUN_CLIMATOLOGY = False
"""

import os
import json
import math
import warnings
import heapq
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import joblib

import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.transform import array_bounds
from shapely.geometry import box

from scipy.ndimage import convolve, binary_dilation
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator


# =============================================================================
# USER CONFIG
# =============================================================================
BODE_DIR = Path(r"C:\goyal_shekhar\post_doc\_for_AQUASUB_ECOSTRESS\Bode_Buffer")

DEM_PATH    = BODE_DIR / "dem_Bode_Buffer.nc"
SLOPE_PATH  = BODE_DIR / "slope_Bode_buffer.nc"
Q_PATH      = BODE_DIR / "Q_2015-2025_Bode_Buffer.nc"
TAVG_PATH   = BODE_DIR / "tavg_2015-2025_Bode_Buffer.nc"
LULC_FILE   = Path(r"Y:\Home\goyal\post_doc_work\ecostress\data\lulc\WorldCover_Bode_2020_30m.tif")

STACK_NC = Path(r"C:\goyal_shekhar\post_doc\_for_AQUASUB_ECOSTRESS\ECO_L2T_LSTE_merged_stack.nc")

# Optional alternate local copy
STACK_NC_LOCAL = None

RIVER_SHP      = Path(r"C:\goyal_shekhar\post_doc\ecostress\data\HydroRIVERS_v10_eu_shp\HydroRIVERS_v10_eu_shp\HydroRIVERS_v10_eu.shp")
BODE_BASIN_SHP = Path(r"C:\goyal_shekhar\post_doc\ecostress\data\bode_shp\bode.shp")

MODEL_PATH = Path(r"C:\goyal_shekhar\post_doc\ecostress\models\rf_full_final.joblib")
FEATS_JSON = Path(r"C:\goyal_shekhar\post_doc\ecostress\models\rf_full_feature_cols.json")

OUTDIR = Path(r"C:\goyal_shekhar\post_doc\ecostress\predictions_rf_full_pubready_projected")
OUTDIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# ECOSTRESS STACK VARIABLE NAMES
# =============================================================================
LAT_DIM  = "ydim"
LON_DIM  = "xdim"
TIME_DIM = "time"
LST_VAR  = "LST_C"


# =============================================================================
# RUN MODES
# =============================================================================
RUN_SINGLE_SCENE = True
RUN_CLIMATOLOGY  = False   # keep False for first successful run

# Single-scene selection
TIME_STR = None
SCAN_N = 400
MIN_VALID_FRAC = 0.10

# Climatology
MAX_SCENES_FOR_CLIMATOLOGY = 100   # set None for all scenes after testing
MIN_SCENE_VALID_FRAC_FOR_CLIM = 0.01


# =============================================================================
# RIVER CORRIDOR + FILL
# =============================================================================
BBOX_PAD_DEG = 0.02
RIVER_DILATE_PX = 4
MAX_FILL_DIST_M = 5000
EIGHT_CONNECTED = True

USE_TEMPORAL_FILL = True
TEMPORAL_MAX_OFFSET = 30


# =============================================================================
# BUFFERS / LULC
# =============================================================================
BUFFER_RADII_M = [50, 100, 150, 200, 300, 400, 500]
WORLDCOVER_CLASSES = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
LULC_RADIUS_M = 500


# =============================================================================
# PLOTTING / EXPORT
# =============================================================================
PLOT_CRS_PROJECTED = "EPSG:25832"
PLOT_CRS_WGS84     = "EPSG:4326"

PCTL_VMIN = 2
PCTL_VMAX = 98
SCALEBAR_KM = 10
CMAP_TEMP = "coolwarm"

PLOT_RIVER_UNDERLAY = False


# =============================================================================
# STYLE
# =============================================================================
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth": 1.0,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 600
})


# =============================================================================
# LOGGING
# =============================================================================
def log(msg):
    print(msg, flush=True)

def ok(msg):
    log(f"✅ {msg}")

def warn(msg):
    log(f"⚠️  {msg}")

def fail(msg):
    raise RuntimeError(f"❌ {msg}")

def require_file(path, label, required=True):
    path = Path(path)
    log(f"[CHECK] {label}: {path}")
    if not path.exists():
        if required:
            fail(f"{label} not found: {path}")
        warn(f"{label} not found (continuing): {path}")
        return False
    ok(f"{label} exists.")
    return True


# =============================================================================
# FILE RESOLUTION
# =============================================================================
def resolve_stack_path():
    if STACK_NC_LOCAL is not None and Path(STACK_NC_LOCAL).exists():
        ok(f"Using local ECOSTRESS stack: {STACK_NC_LOCAL}")
        return Path(STACK_NC_LOCAL)
    ok(f"Using ECOSTRESS stack: {STACK_NC}")
    return Path(STACK_NC)


# =============================================================================
# SAFE NETCDF OPENING
# =============================================================================
def open_nc_safe(path, drop_variables=None):
    """
    Try HDF5-backed netCDF first (h5netcdf), then classic NetCDF3 (scipy),
    then netcdf4 last.
    """
    path = str(path)

    # 1) HDF5 / netCDF4-style files
    try:
        ds = xr.open_dataset(
            path,
            engine="h5netcdf",
            cache=False,
            drop_variables=drop_variables
        )
        ok(f"Opened with h5netcdf: {path}")
        return ds
    except Exception as e:
        warn(f"h5netcdf failed for {path}, trying scipy. Reason: {e}")

    # 2) Classic NetCDF3 files
    try:
        ds = xr.open_dataset(
            path,
            engine="scipy",
            cache=False,
            drop_variables=drop_variables
        )
        ok(f"Opened with scipy: {path}")
        return ds
    except Exception as e:
        warn(f"scipy failed for {path}, trying netcdf4. Reason: {e}")

    # 3) netCDF4 backend last
    ds = xr.open_dataset(
        path,
        engine="netcdf4",
        cache=False,
        drop_variables=drop_variables
    )
    ok(f"Opened with netcdf4: {path}")
    return ds


# =============================================================================
# TIME / SEASON HELPERS
# =============================================================================
def season_from_month(m):
    if m in (12, 1, 2):
        return "DJF"
    if m in (3, 4, 5):
        return "MAM"
    if m in (6, 7, 8):
        return "JJA"
    return "SON"

def season_window(ts):
    ts = pd.Timestamp(ts)
    y = ts.year
    m = ts.month

    if m == 12:
        return "DJF", pd.Timestamp(y, 12, 1), pd.Timestamp(y + 1, 2, 28) + pd.offsets.MonthEnd(0)
    elif m in (1, 2):
        return "DJF", pd.Timestamp(y - 1, 12, 1), pd.Timestamp(y, 2, 28) + pd.offsets.MonthEnd(0)
    elif m in (3, 4, 5):
        return "MAM", pd.Timestamp(y, 3, 1), pd.Timestamp(y, 5, 31)
    elif m in (6, 7, 8):
        return "JJA", pd.Timestamp(y, 6, 1), pd.Timestamp(y, 8, 31)
    else:
        return "SON", pd.Timestamp(y, 9, 1), pd.Timestamp(y, 11, 30)

def build_time_features(ts, n):
    ts = pd.Timestamp(ts)
    doy = float(ts.dayofyear)
    hour = float(ts.hour + ts.minute / 60.0)
    month = float(ts.month)
    season = season_from_month(ts.month)

    return {
        "doy_sin": np.full(n, np.sin(2 * np.pi * doy / 365.25), dtype=np.float32),
        "doy_cos": np.full(n, np.cos(2 * np.pi * doy / 365.25), dtype=np.float32),
        "month":   np.full(n, month, dtype=np.float32),
        "month_sin": np.full(n, np.sin(2 * np.pi * month / 12.0), dtype=np.float32),
        "month_cos": np.full(n, np.cos(2 * np.pi * month / 12.0), dtype=np.float32),
        "hour":    np.full(n, hour, dtype=np.float32),
        "is_day":  np.full(n, 1.0 if 6 <= hour < 18 else 0.0, dtype=np.float32),
        "season_DJF": np.full(n, 1.0 if season == "DJF" else 0.0, dtype=np.float32),
        "season_MAM": np.full(n, 1.0 if season == "MAM" else 0.0, dtype=np.float32),
        "season_JJA": np.full(n, 1.0 if season == "JJA" else 0.0, dtype=np.float32),
        "season_SON": np.full(n, 1.0 if season == "SON" else 0.0, dtype=np.float32),
    }


# =============================================================================
# GRID / RASTER HELPERS
# =============================================================================
def kernel_circle(dx_m, dy_m, radius_m):
    rx = int(np.ceil(radius_m / max(dx_m, 1e-6)))
    ry = int(np.ceil(radius_m / max(dy_m, 1e-6)))
    yy, xx = np.mgrid[-ry:ry+1, -rx:rx+1]
    dist = np.sqrt((yy * dy_m) ** 2 + (xx * dx_m) ** 2)
    return (dist <= radius_m).astype(np.float32)

def masked_mean_conv(values2d, valid2d, kernel):
    v = np.where(valid2d, values2d, 0.0).astype(np.float32)
    c = valid2d.astype(np.float32)
    s = convolve(v, kernel, mode="constant", cval=0.0)
    n = convolve(c, kernel, mode="constant", cval=0.0)

    out = np.full(values2d.shape, np.nan, dtype=np.float32)
    msk = n > 0
    out[msk] = s[msk] / n[msk]
    return out

def reproject_tif_to_grid(src_tif, dst_shape, dst_transform, dst_crs="EPSG:4326", is_categorical=False):
    with rasterio.open(src_tif) as src:
        src_arr = src.read(1)
        dst = np.zeros(dst_shape, dtype=np.float32)

        reproject(
            source=src_arr,
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,
            resampling=Resampling.nearest if is_categorical else Resampling.bilinear,
        )
        return dst

def pick_data_var(ds, preferred=()):
    for p in preferred:
        if p in ds.data_vars:
            return p
    dvs = list(ds.data_vars)
    if len(dvs) == 0:
        fail("No data variables in dataset.")
    return dvs[0]

def find_lat_lon_names(ds):
    cand_lat = [n for n in list(ds.coords) + list(ds.dims) if n.lower() in ("lat", "latitude", "y", "ydim")]
    cand_lon = [n for n in list(ds.coords) + list(ds.dims) if n.lower() in ("lon", "longitude", "x", "xdim")]
    lat_name = cand_lat[0] if cand_lat else None
    lon_name = cand_lon[0] if cand_lon else None

    if lat_name is None or lon_name is None:
        for n in ds.coords:
            v = ds[n].values
            if v.ndim == 1 and np.issubdtype(v.dtype, np.number):
                if lat_name is None and np.nanmin(v) >= -90 and np.nanmax(v) <= 90:
                    lat_name = n
                if lon_name is None and np.nanmin(v) >= -180 and np.nanmax(v) <= 180:
                    lon_name = n
    return lat_name, lon_name

def interp_da_to_grid(da, lat_name, lon_name, target_lat, target_lon, method="nearest"):
    da = da.squeeze(drop=True)

    if lat_name not in da.dims or lon_name not in da.dims:
        raise RuntimeError(
            f"interp_da_to_grid expected dims '{lat_name}' and '{lon_name}', got dims={da.dims}"
        )

    da = da.load()

    lat_vals = da[lat_name].values
    lon_vals = da[lon_name].values

    if np.any(np.diff(lat_vals) < 0):
        da = da.sortby(lat_name)
    if np.any(np.diff(lon_vals) < 0):
        da = da.sortby(lon_name)

    target_lat_da = xr.DataArray(target_lat, dims=(LAT_DIM,))
    target_lon_da = xr.DataArray(target_lon, dims=(LON_DIM,))

    out = da.interp(
        {lat_name: target_lat_da, lon_name: target_lon_da},
        method=method
    ).load()

    return out.values.astype(np.float32)


# =============================================================================
# SAFE SCENE READING
# =============================================================================
def read_ecostress_scene_safe(ds, ti, stack_path=None, var=LST_VAR):
    errors = []

    try:
        a = ds[var].isel({TIME_DIM: ti}).values
        return np.asarray(a, dtype=np.float32)
    except Exception as e:
        errors.append(f"open-ds read failed: {e}")

    try:
        a = ds[var].isel({TIME_DIM: ti}).load().values
        return np.asarray(a, dtype=np.float32)
    except Exception as e:
        errors.append(f"open-ds load failed: {e}")

    if stack_path is not None:
        for engine in ["h5netcdf", "scipy", "netcdf4"]:
            try:
                ds_tmp = xr.open_dataset(
                    str(stack_path),
                    engine=engine,
                    cache=False,
                    drop_variables=["QC", "CLOUD", "WATER"]
                )
                a = ds_tmp[var].isel({TIME_DIM: ti}).values
                a = np.asarray(a, dtype=np.float32)
                ds_tmp.close()
                return a
            except Exception as e:
                errors.append(f"reopen with {engine} failed: {e}")

    raise RuntimeError(" | ".join(errors))


# =============================================================================
# STREAM-ONLY GAP FILL
# =============================================================================
def fill_lst_along_stream(lst2d, mask, dx_m, dy_m, max_dist_m=5000, eight_connected=True):
    H, W = mask.shape
    ys, xs = np.where(mask)
    n_nodes = ys.size
    if n_nodes == 0:
        return lst2d.astype(np.float32), np.zeros_like(mask, dtype=bool)

    node_id = np.full((H, W), -1, dtype=np.int32)
    node_id[ys, xs] = np.arange(n_nodes, dtype=np.int32)

    valid = np.isfinite(lst2d) & mask
    v_y, v_x = np.where(valid)
    sources = node_id[v_y, v_x]
    sources = sources[sources >= 0]
    if sources.size == 0:
        return lst2d.astype(np.float32), np.zeros_like(mask, dtype=bool)

    dist = np.full(n_nodes, np.inf, dtype=np.float32)
    src  = np.full(n_nodes, -1, dtype=np.int32)
    heap = []

    for s in sources:
        dist[s] = 0.0
        src[s] = s
        heapq.heappush(heap, (0.0, int(s)))

    moves = [(-1, 0, float(dy_m)), (1, 0, float(dy_m)), (0, -1, float(dx_m)), (0, 1, float(dx_m))]
    if eight_connected:
        ddiag = float(np.hypot(dx_m, dy_m))
        moves += [(-1, -1, ddiag), (-1, 1, ddiag), (1, -1, ddiag), (1, 1, ddiag)]

    maxd = float(max_dist_m)

    while heap:
        d, u = heapq.heappop(heap)
        if d != dist[u]:
            continue
        if d > maxd:
            break

        y = int(ys[u])
        x = int(xs[u])
        su = int(src[u])

        for oy, ox, w in moves:
            yy = y + oy
            xx = x + ox
            if yy < 0 or yy >= H or xx < 0 or xx >= W:
                continue
            v = int(node_id[yy, xx])
            if v < 0:
                continue
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                src[v] = su
                heapq.heappush(heap, (float(nd), v))

    lst_filled = lst2d.astype(np.float32).copy()
    filled_flag = np.zeros((H, W), dtype=bool)

    missing = mask & (~np.isfinite(lst2d))
    my, mx = np.where(missing)
    if my.size > 0:
        miss_nodes = node_id[my, mx]
        keep = miss_nodes >= 0
        miss_nodes = miss_nodes[keep]
        my = my[keep]
        mx = mx[keep]

        if miss_nodes.size > 0:
            good = np.isfinite(dist[miss_nodes]) & (dist[miss_nodes] <= maxd) & (src[miss_nodes] >= 0)
            if np.any(good):
                tgt_nodes = miss_nodes[good]
                ty = my[good]
                tx = mx[good]
                src_nodes = src[tgt_nodes]
                sy = ys[src_nodes].astype(int)
                sx = xs[src_nodes].astype(int)
                lst_filled[ty, tx] = lst2d[sy, sx].astype(np.float32)
                filled_flag[ty, tx] = True

    return lst_filled, filled_flag

def temporal_fill_same_pixel(ds, ti0, mask, lst2d_in, max_offset=30, lst_var="LST_C", stack_path=None):
    lst = lst2d_in.astype(np.float32).copy()
    filled = np.zeros(mask.shape, dtype=bool)

    miss = mask & (~np.isfinite(lst))
    if miss.sum() == 0:
        return lst, filled

    nt = ds.sizes[TIME_DIM]

    for k in range(1, max_offset + 1):
        changed = 0
        for sign in (-1, +1):
            ti = ti0 + sign * k
            if ti < 0 or ti >= nt:
                continue

            miss = mask & (~np.isfinite(lst))
            if miss.sum() == 0:
                break

            try:
                a = read_ecostress_scene_safe(ds, ti, stack_path=stack_path, var=lst_var)
            except Exception:
                continue

            can = miss & np.isfinite(a)
            if np.any(can):
                lst[can] = a[can]
                filled[can] = True
                changed += int(can.sum())

        if (mask & (~np.isfinite(lst))).sum() == 0:
            break
        if changed == 0:
            continue

    return lst, filled


# =============================================================================
# MAP PROJECTION / PLOT HELPERS
# =============================================================================
def extent_from_transform(transform, width, height):
    left, bottom, right, top = array_bounds(height, width, transform)
    return [left, right, bottom, top]

def reproject_array_for_plot(arr2d, src_transform, src_crs, dst_crs, resampling=Resampling.bilinear):
    h, w = arr2d.shape
    left, bottom, right, top = array_bounds(h, w, src_transform)

    dst_transform, dst_width, dst_height = calculate_default_transform(
        src_crs, dst_crs, w, h, left, bottom, right, top
    )

    dst = np.full((dst_height, dst_width), np.nan, dtype=np.float32)

    reproject(
        source=arr2d.astype(np.float32),
        destination=dst,
        src_transform=src_transform,
        src_crs=src_crs,
        src_nodata=np.nan,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        dst_nodata=np.nan,
        resampling=resampling
    )
    return dst, dst_transform

def add_north_arrow(ax, x=0.93, y=0.90, size=0.09):
    ax.annotate(
        "N",
        xy=(x, y), xycoords="axes fraction",
        xytext=(x, y - size), textcoords="axes fraction",
        ha="center", va="center",
        fontsize=12, fontweight="bold",
        arrowprops=dict(arrowstyle="-|>", lw=1.2, color="black")
    )

def add_scalebar_projected(ax, length_km=10, loc=(0.08, 0.06), height_frac=0.012, ndiv=2):
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    x0 = xmin + loc[0] * (xmax - xmin)
    y0 = ymin + loc[1] * (ymax - ymin)
    bar_h = height_frac * (ymax - ymin)
    length_m = length_km * 1000.0
    seg = length_m / ndiv

    for i in range(ndiv):
        face = "black" if i % 2 == 0 else "white"
        ax.add_patch(Rectangle(
            (x0 + i * seg, y0), seg, bar_h,
            facecolor=face, edgecolor="black", lw=0.8, zorder=30
        ))

    ax.text(
        x0 + length_m / 2, y0 + 1.6 * bar_h, f"{length_km} km",
        ha="center", va="bottom", fontsize=10, zorder=31
    )

def add_scalebar_wgs84(ax, length_km=10, ndiv=2, loc=(0.08, 0.06)):
    lon_min, lon_max = ax.get_xlim()
    lat_min, lat_max = ax.get_ylim()

    lat0 = 0.5 * (lat_min + lat_max)
    deg_per_km_lon = 1.0 / (111.32 * np.cos(np.deg2rad(lat0)))
    length_deg = length_km * deg_per_km_lon

    x0 = lon_min + loc[0] * (lon_max - lon_min)
    y0 = lat_min + loc[1] * (lat_max - lat_min)
    bar_h = 0.012 * (lat_max - lat_min)
    seg = length_deg / ndiv

    for i in range(ndiv):
        face = "black" if i % 2 == 0 else "white"
        ax.add_patch(Rectangle(
            (x0 + i * seg, y0), seg, bar_h,
            facecolor=face, edgecolor="black", lw=0.8, zorder=30
        ))

    ax.text(
        x0 + length_deg / 2, y0 + 1.6 * bar_h, f"{length_km} km",
        ha="center", va="bottom", fontsize=10, zorder=31
    )

def zoom_to_basin(ax, basin_gdf, pad_frac=0.05):
    bminx, bminy, bmaxx, bmaxy = basin_gdf.total_bounds
    padx = pad_frac * (bmaxx - bminx)
    pady = pad_frac * (bmaxy - bminy)
    ax.set_xlim(bminx - padx, bmaxx + padx)
    ax.set_ylim(bminy - pady, bmaxy + pady)

def save_float32_geotiff(out_tif, arr2d, transform, crs="EPSG:4326", nodata=-9999.0):
    out_tif = str(out_tif)
    h, w = arr2d.shape
    write_arr = np.where(np.isfinite(arr2d), arr2d, nodata).astype(np.float32)

    with rasterio.open(
        out_tif,
        "w",
        driver="GTiff",
        height=h,
        width=w,
        count=1,
        dtype="float32",
        crs=crs,
        transform=transform,
        nodata=nodata
    ) as dst:
        dst.write(write_arr, 1)

def make_single_map(arr2d, transform, basin_gdf, river_gdf, out_png, title,
                    dst_crs="EPSG:25832", cmap="coolwarm",
                    vmin=None, vmax=None, plot_river_underlay=False):
    if dst_crs == "EPSG:4326":
        plot_arr = arr2d.copy()
        plot_transform = transform
    else:
        plot_arr, plot_transform = reproject_array_for_plot(
            arr2d, transform, "EPSG:4326", dst_crs, resampling=Resampling.bilinear
        )

    basin_p = basin_gdf.to_crs(dst_crs)
    river_p = river_gdf.to_crs(dst_crs)

    finite_vals = plot_arr[np.isfinite(plot_arr)]
    if finite_vals.size == 0:
        fail("No finite values available for plotting.")

    if vmin is None:
        vmin = float(np.nanpercentile(finite_vals, PCTL_VMIN))
    if vmax is None:
        vmax = float(np.nanpercentile(finite_vals, PCTL_VMAX))

    ext = extent_from_transform(plot_transform, plot_arr.shape[1], plot_arr.shape[0])

    fig, ax = plt.subplots(figsize=(10.5, 8.0))

    im = ax.imshow(
        np.ma.masked_invalid(plot_arr),
        origin="upper",
        extent=ext,
        interpolation="nearest",
        cmap=cmap,
        norm=Normalize(vmin=vmin, vmax=vmax),
        zorder=2
    )

    if plot_river_underlay and not river_p.empty:
        river_p.plot(ax=ax, linewidth=0.45, color="0.75", alpha=0.9, zorder=3)

    basin_p.boundary.plot(ax=ax, linewidth=1.5, color="black", alpha=0.95, zorder=5)

    cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cb.set_label("Predicted river water temperature (°C)")
    cb.locator = MaxNLocator(nbins=6)
    cb.update_ticks()

    zoom_to_basin(ax, basin_p, pad_frac=0.05)

    ax.set_title(title, pad=10)
    ax.grid(False)
    ax.tick_params(direction="out", length=4, width=1)

    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)

    if dst_crs == "EPSG:4326":
        ax.set_xlabel("Longitude (°E)")
        ax.set_ylabel("Latitude (°N)")
        add_north_arrow(ax, x=0.93, y=0.90, size=0.09)
        add_scalebar_wgs84(ax, length_km=SCALEBAR_KM, ndiv=2, loc=(0.07, 0.06))
    else:
        ax.set_xlabel("Easting (m)")
        ax.set_ylabel("Northing (m)")
        add_north_arrow(ax, x=0.93, y=0.90, size=0.09)
        add_scalebar_projected(ax, length_km=SCALEBAR_KM, ndiv=2, loc=(0.07, 0.06))

    plt.tight_layout()
    fig.savefig(out_png, dpi=600, bbox_inches="tight", pad_inches=0.02)
    plt.show()
    plt.close(fig)

    return vmin, vmax

def make_climatology_panel(maps_dict, transform, basin_gdf, river_gdf, out_png,
                           title="Long-term mean predicted river water temperature",
                           dst_crs="EPSG:25832", cmap="coolwarm",
                           plot_river_underlay=False):
    order = ["Annual", "DJF", "MAM", "JJA", "SON"]
    plot_arrays = {}
    plot_transform = None

    for key in order:
        arr = maps_dict[key]
        if dst_crs == "EPSG:4326":
            plot_arrays[key] = arr.copy()
            plot_transform = transform
        else:
            prj, plot_transform = reproject_array_for_plot(
                arr, transform, "EPSG:4326", dst_crs, resampling=Resampling.bilinear
            )
            plot_arrays[key] = prj

    basin_p = basin_gdf.to_crs(dst_crs)
    river_p = river_gdf.to_crs(dst_crs)

    all_vals = np.concatenate([
        plot_arrays[k][np.isfinite(plot_arrays[k])]
        for k in order if np.isfinite(plot_arrays[k]).any()
    ])
    vmin = float(np.nanpercentile(all_vals, PCTL_VMIN))
    vmax = float(np.nanpercentile(all_vals, PCTL_VMAX))

    fig, axes = plt.subplots(2, 3, figsize=(13.5, 9.2))
    axes = axes.ravel()

    last_im = None
    for i, key in enumerate(order):
        ax = axes[i]
        arr = plot_arrays[key]
        ext = extent_from_transform(plot_transform, arr.shape[1], arr.shape[0])

        last_im = ax.imshow(
            np.ma.masked_invalid(arr),
            origin="upper",
            extent=ext,
            interpolation="nearest",
            cmap=cmap,
            norm=Normalize(vmin=vmin, vmax=vmax),
            zorder=2
        )

        if plot_river_underlay and not river_p.empty:
            river_p.plot(ax=ax, linewidth=0.35, color="0.78", alpha=0.9, zorder=3)

        basin_p.boundary.plot(ax=ax, linewidth=1.0, color="black", alpha=0.95, zorder=5)
        zoom_to_basin(ax, basin_p, pad_frac=0.05)

        ax.set_title(key, pad=6)
        ax.grid(False)
        ax.tick_params(direction="out", length=3.5, width=0.9)

        for sp in ["top", "right"]:
            ax.spines[sp].set_visible(False)

        if dst_crs == "EPSG:4326":
            ax.set_xlabel("Longitude (°E)")
            ax.set_ylabel("Latitude (°N)")
        else:
            ax.set_xlabel("Easting (m)")
            ax.set_ylabel("Northing (m)")

    axes[5].axis("off")

    cbar = fig.colorbar(last_im, ax=axes.tolist(), fraction=0.025, pad=0.02)
    cbar.set_label("Predicted river water temperature (°C)")
    cbar.locator = MaxNLocator(nbins=6)
    cbar.update_ticks()

    fig.suptitle(title, y=0.98, fontsize=14)

    if dst_crs == "EPSG:4326":
        add_north_arrow(axes[0], x=0.93, y=0.88, size=0.08)
        add_scalebar_wgs84(axes[0], length_km=SCALEBAR_KM, ndiv=2, loc=(0.07, 0.06))
    else:
        add_north_arrow(axes[0], x=0.93, y=0.88, size=0.08)
        add_scalebar_projected(axes[0], length_km=SCALEBAR_KM, ndiv=2, loc=(0.07, 0.06))

    plt.tight_layout(rect=[0, 0, 1, 0.965])
    fig.savefig(out_png, dpi=600, bbox_inches="tight", pad_inches=0.02)
    plt.show()
    plt.close(fig)

    return vmin, vmax


# =============================================================================
# DATA PREPARATION
# =============================================================================
def pick_time_index(ds, times, stack_path):
    if TIME_STR is not None:
        t0 = pd.to_datetime(TIME_STR)
        idx = int(np.argmin(np.abs(times - t0)))
        ok(f"Using TIME_STR={TIME_STR} -> nearest scene = {times[idx]}")
        return idx

    ok(f"Auto-picking best-valid scene among first {min(SCAN_N, len(times))} scenes...")
    best_v = -1.0
    best_i = None
    N = min(SCAN_N, len(times))

    for i in range(N):
        try:
            a = read_ecostress_scene_safe(ds, i, stack_path=stack_path, var=LST_VAR)
            v = float(np.isfinite(a).mean())
        except Exception as e:
            warn(f"Could not read scene {i} ({times[i]}): {e}")
            continue

        if v > best_v:
            best_v = v
            best_i = i
        if v >= MIN_VALID_FRAC:
            ok(f"Early pick: valid fraction={v:.3f} at {times[i]}")
            return i

    if best_i is None:
        fail("Could not read any ECOSTRESS scene successfully.")

    ok(f"Best among scanned: valid fraction={best_v:.3f} at {times[best_i]}")
    return best_i

def prepare_context():
    stack_path = resolve_stack_path()

    require_file(MODEL_PATH, "MODEL_PATH")
    require_file(FEATS_JSON, "FEATS_JSON")
    require_file(stack_path, "STACK_NC")
    require_file(RIVER_SHP, "RIVER_SHP")
    require_file(DEM_PATH, "DEM_PATH")
    require_file(SLOPE_PATH, "SLOPE_PATH")
    require_file(Q_PATH, "Q_PATH")
    require_file(TAVG_PATH, "TAVG_PATH")
    require_file(LULC_FILE, "LULC_FILE")
    require_file(BODE_BASIN_SHP, "BODE_BASIN_SHP")

    model = joblib.load(MODEL_PATH)
    with open(FEATS_JSON, "r") as f:
        feature_cols = json.load(f)
    ok(f"Loaded model + {len(feature_cols)} features.")

    ds = open_nc_safe(stack_path, drop_variables=["QC", "CLOUD", "WATER"])
    lat = ds[LAT_DIM].values
    lon = ds[LON_DIM].values
    times = pd.to_datetime(ds[TIME_DIM].values)

    H, W = len(lat), len(lon)
    dy_deg = float(abs(lat[1] - lat[0]))
    dx_deg = float(abs(lon[1] - lon[0]))
    lat0 = float(np.nanmean(lat))
    dx_m = dx_deg * 111_320.0 * math.cos(math.radians(lat0))
    dy_m = dy_deg * 111_320.0

    lon_min, lon_max = float(lon.min()), float(lon.max())
    lat_min, lat_max = float(lat.min()), float(lat.max())
    transform = from_origin(lon_min - dx_deg / 2, lat_max + dy_deg / 2, dx_deg, dy_deg)

    ok(f"ECOSTRESS grid: HxW={H}x{W}")
    ok(f"Approx resolution: dx={dx_m:.1f} m, dy={dy_m:.1f} m")
    ok(f"Time range: {times.min()} to {times.max()} (n={len(times)})")

    basin = gpd.read_file(BODE_BASIN_SHP).to_crs("EPSG:4326")
    basin_mask = rasterize(
        [(geom, 1) for geom in basin.geometry if geom is not None and not geom.is_empty],
        out_shape=(H, W),
        transform=transform,
        fill=0,
        all_touched=True,
        dtype="uint8"
    ).astype(bool)

    river = gpd.read_file(RIVER_SHP).to_crs("EPSG:4326")
    bbox_geom = box(lon_min - BBOX_PAD_DEG, lat_min - BBOX_PAD_DEG,
                    lon_max + BBOX_PAD_DEG, lat_max + BBOX_PAD_DEG)
    river_clip_bbox = river[river.intersects(bbox_geom)].copy()
    if river_clip_bbox.empty:
        fail("No rivers found in bbox.")

    river_mask = rasterize(
        [(geom, 1) for geom in river_clip_bbox.geometry if geom is not None and not geom.is_empty],
        out_shape=(H, W),
        transform=transform,
        fill=0,
        all_touched=True,
        dtype="uint8"
    ).astype(bool)

    river_corridor = binary_dilation(river_mask, iterations=int(RIVER_DILATE_PX))
    pred_domain = basin_mask & river_corridor

    try:
        river_in_basin = gpd.clip(river_clip_bbox, basin)
    except Exception:
        river_in_basin = river_clip_bbox.copy()

    ds_dem = open_nc_safe(DEM_PATH)
    v_dem = pick_data_var(ds_dem, preferred=("dem", "DEM", "Band1"))
    lat_dem, lon_dem = find_lat_lon_names(ds_dem)
    dem2d = interp_da_to_grid(ds_dem[v_dem], lat_dem, lon_dem, lat, lon, method="nearest")

    ds_slope = open_nc_safe(SLOPE_PATH)
    v_slope = pick_data_var(ds_slope, preferred=("slope", "SLOPE", "Band1"))
    lat_slope, lon_slope = find_lat_lon_names(ds_slope)
    slope2d = interp_da_to_grid(ds_slope[v_slope], lat_slope, lon_slope, lat, lon, method="nearest")

    lulc_grid = reproject_tif_to_grid(
        str(LULC_FILE), (H, W), transform, dst_crs="EPSG:4326", is_categorical=True
    ).astype(np.int32)

    k500 = kernel_circle(dx_m, dy_m, LULC_RADIUS_M)
    valid_lulc = np.isfinite(lulc_grid) & (lulc_grid != 0)
    count_total = convolve(valid_lulc.astype(np.float32), k500, mode="constant", cval=0.0)
    count_total[count_total == 0] = np.nan

    lulc_frac = {}
    for cls in WORLDCOVER_CLASSES:
        cnt = convolve(((lulc_grid == cls) & valid_lulc).astype(np.float32), k500, mode="constant", cval=0.0)
        lulc_frac[f"lulc_{cls}"] = (cnt / count_total).astype(np.float32)

    dsQ = open_nc_safe(Q_PATH)
    vQ = pick_data_var(dsQ, preferred=("Q", "q", "discharge"))
    daQ = dsQ[vQ]
    tdimQ = [d for d in daQ.dims if "time" in d.lower()][0]
    latQ, lonQ = find_lat_lon_names(dsQ)
    tq = pd.to_datetime(dsQ[tdimQ].values)

    dsT = open_nc_safe(TAVG_PATH)
    vT = pick_data_var(dsT, preferred=("tavg", "T_air", "tas"))
    daT = dsT[vT]
    tdimT = [d for d in daT.dims if "time" in d.lower()][0]
    latT, lonT = find_lat_lon_names(dsT)
    tt = pd.to_datetime(dsT[tdimT].values)

    return {
        "stack_path": stack_path,
        "model": model,
        "feature_cols": feature_cols,
        "ds": ds,
        "lat": lat,
        "lon": lon,
        "times": times,
        "H": H,
        "W": W,
        "dx_deg": dx_deg,
        "dy_deg": dy_deg,
        "dx_m": dx_m,
        "dy_m": dy_m,
        "transform": transform,
        "basin": basin,
        "basin_mask": basin_mask,
        "river_clip_bbox": river_clip_bbox,
        "river_in_basin": river_in_basin,
        "river_mask": river_mask,
        "river_corridor": river_corridor,
        "pred_domain": pred_domain,
        "dem2d": dem2d,
        "slope2d": slope2d,
        "lulc_frac": lulc_frac,
        "dsQ": dsQ,
        "vQ": vQ,
        "daQ": daQ,
        "tdimQ": tdimQ,
        "latQ": latQ,
        "lonQ": lonQ,
        "tq": tq,
        "dsT": dsT,
        "vT": vT,
        "daT": daT,
        "tdimT": tdimT,
        "latT": latT,
        "lonT": lonT,
        "tt": tt
    }


# =============================================================================
# SCENE PREDICTION
# =============================================================================
def get_current_month_and_season_grids(ctx, t0):
    lat = ctx["lat"]
    lon = ctx["lon"]

    daQ = ctx["daQ"]
    daT = ctx["daT"]

    Q_now = daQ.sel({ctx["tdimQ"]: t0}, method="nearest").load()
    T_now = daT.sel({ctx["tdimT"]: t0}, method="nearest").load()

    Q_now2d = interp_da_to_grid(Q_now, ctx["latQ"], ctx["lonQ"], lat, lon, method="nearest")
    T_now2d = interp_da_to_grid(T_now, ctx["latT"], ctx["lonT"], lat, lon, method="nearest")

    ts = pd.Timestamp(t0)
    month_start = ts.to_period("M").start_time
    month_end   = ts.to_period("M").end_time

    daQ_m = daQ.sel({ctx["tdimQ"]: slice(month_start, month_end)})
    daT_m = daT.sel({ctx["tdimT"]: slice(month_start, month_end)})

    Q_month_mean = daQ_m.mean(dim=ctx["tdimQ"]).load()
    Q_month_max  = daQ_m.max(dim=ctx["tdimQ"]).load()
    T_month_mean = daT_m.mean(dim=ctx["tdimT"]).load()
    T_month_max  = daT_m.max(dim=ctx["tdimT"]).load()

    Q_month_mean2d = interp_da_to_grid(Q_month_mean, ctx["latQ"], ctx["lonQ"], lat, lon, method="nearest")
    Q_month_max2d  = interp_da_to_grid(Q_month_max,  ctx["latQ"], ctx["lonQ"], lat, lon, method="nearest")
    T_month_mean2d = interp_da_to_grid(T_month_mean, ctx["latT"], ctx["lonT"], lat, lon, method="nearest")
    T_month_max2d  = interp_da_to_grid(T_month_max,  ctx["latT"], ctx["lonT"], lat, lon, method="nearest")

    season, season_start, season_end = season_window(ts)

    daQ_s = daQ.sel({ctx["tdimQ"]: slice(season_start, season_end)})
    daT_s = daT.sel({ctx["tdimT"]: slice(season_start, season_end)})

    Q_season_mean = daQ_s.mean(dim=ctx["tdimQ"]).load()
    Q_season_max  = daQ_s.max(dim=ctx["tdimQ"]).load()
    T_season_mean = daT_s.mean(dim=ctx["tdimT"]).load()
    T_season_max  = daT_s.max(dim=ctx["tdimT"]).load()

    Q_season_mean2d = interp_da_to_grid(Q_season_mean, ctx["latQ"], ctx["lonQ"], lat, lon, method="nearest")
    Q_season_max2d  = interp_da_to_grid(Q_season_max,  ctx["latQ"], ctx["lonQ"], lat, lon, method="nearest")
    T_season_mean2d = interp_da_to_grid(T_season_mean, ctx["latT"], ctx["lonT"], lat, lon, method="nearest")
    T_season_max2d  = interp_da_to_grid(T_season_max,  ctx["latT"], ctx["lonT"], lat, lon, method="nearest")

    return {
        "Q": Q_now2d,
        "T_air": T_now2d,
        "Q_month_mean": Q_month_mean2d,
        "Q_month_max": Q_month_max2d,
        "T_month_mean": T_month_mean2d,
        "T_month_max": T_month_max2d,
        "Q_season_mean": Q_season_mean2d,
        "Q_season_max": Q_season_max2d,
        "T_season_mean": T_season_mean2d,
        "T_season_max": T_season_max2d,
        "season_label": season
    }

def predict_scene(ctx, ti0):
    ds = ctx["ds"]
    t0 = ctx["times"][ti0]
    model = ctx["model"]
    feature_cols = ctx["feature_cols"]

    H = ctx["H"]
    W = ctx["W"]
    pred_domain = ctx["pred_domain"]

    try:
        lst2d = read_ecostress_scene_safe(ds, ti0, stack_path=ctx["stack_path"], var=LST_VAR)
    except Exception as e:
        warn(f"Failed to read ECOSTRESS scene {ti0} at {t0}: {e}")
        return {
            "time": t0,
            "Tw_plot": np.full((H, W), np.nan, dtype=np.float32),
            "n_target": 0,
            "valid_scene_frac": np.nan,
            "filled_stream": 0,
            "filled_temp": 0
        }

    vfrac_scene = float(np.isfinite(lst2d).mean())

    fill_mask_stream = ctx["river_corridor"]

    lst_streamfilled, flag_stream = fill_lst_along_stream(
        lst2d=lst2d,
        mask=fill_mask_stream,
        dx_m=ctx["dx_m"],
        dy_m=ctx["dy_m"],
        max_dist_m=MAX_FILL_DIST_M,
        eight_connected=EIGHT_CONNECTED
    )

    if USE_TEMPORAL_FILL:
        lst_final, flag_temp = temporal_fill_same_pixel(
            ds=ds,
            ti0=ti0,
            mask=fill_mask_stream,
            lst2d_in=lst_streamfilled,
            max_offset=TEMPORAL_MAX_OFFSET,
            lst_var=LST_VAR,
            stack_path=ctx["stack_path"]
        )
    else:
        lst_final = lst_streamfilled
        flag_temp = np.zeros(fill_mask_stream.shape, dtype=bool)

    target_mask = pred_domain & np.isfinite(lst_final)
    flat_idx = np.flatnonzero(target_mask)
    n = flat_idx.size

    if n == 0:
        return {
            "time": t0,
            "Tw_plot": np.full((H, W), np.nan, dtype=np.float32),
            "n_target": 0,
            "valid_scene_frac": vfrac_scene,
            "filled_stream": int(flag_stream.sum()),
            "filled_temp": int(flag_temp.sum())
        }

    dyn = get_current_month_and_season_grids(ctx, t0)

    valid_for_ts = pred_domain & np.isfinite(lst_final)

    k3 = np.ones((3, 3), dtype=np.float32)
    center_mean2d = masked_mean_conv(lst_final, valid_for_ts, k3)

    buf_mean2d = {}
    for r in BUFFER_RADII_M:
        k = kernel_circle(ctx["dx_m"], ctx["dy_m"], r)
        buf_mean2d[f"buf{r}m_mean"] = masked_mean_conv(lst_final, valid_for_ts, k)

    for r in (50, 100, 150):
        key = f"buf{r}m_mean"
        m = ~np.isfinite(buf_mean2d[key]) & np.isfinite(center_mean2d)
        buf_mean2d[key][m] = center_mean2d[m]

    feats = {}

    feats["center_mean"] = center_mean2d.ravel()[flat_idx]
    for r in BUFFER_RADII_M:
        feats[f"buf{r}m_mean"] = buf_mean2d[f"buf{r}m_mean"].ravel()[flat_idx]

    for k in [
        "Q", "T_air",
        "Q_month_mean", "Q_month_max",
        "T_month_mean", "T_month_max",
        "Q_season_mean", "Q_season_max",
        "T_season_mean", "T_season_max"
    ]:
        feats[k] = dyn[k].ravel()[flat_idx]

    feats["DEM_m"] = ctx["dem2d"].ravel()[flat_idx]
    feats["slope_deg_or_pct"] = ctx["slope2d"].ravel()[flat_idx]

    for cls in WORLDCOVER_CLASSES:
        feats[f"lulc_{cls}"] = ctx["lulc_frac"][f"lulc_{cls}"].ravel()[flat_idx]

    feats.update(build_time_features(t0, n))

    feats["center_valid_count"] = np.isfinite(center_mean2d).ravel()[flat_idx].astype(np.float32)
    for r in BUFFER_RADII_M:
        feats[f"buf{r}m_valid_count"] = np.isfinite(buf_mean2d[f"buf{r}m_mean"]).ravel()[flat_idx].astype(np.float32)

    unsupported = [c for c in feature_cols if c.startswith("T_air_lag") or c.startswith("Q_lag")]
    if unsupported:
        fail(f"Model expects lagged predictors not constructed in this mapping workflow: {unsupported}")

    missing = [c for c in feature_cols if c not in feats]
    if missing:
        fail(f"Missing required model features: {missing}")

    X = np.column_stack([feats[c] for c in feature_cols]).astype(np.float32)
    good = np.isfinite(X).all(axis=1)

    ypred = np.full(n, np.nan, dtype=np.float32)
    if np.any(good):
        ypred[good] = model.predict(X[good]).astype(np.float32)

    Tw_flat = np.full(H * W, np.nan, dtype=np.float32)
    Tw_flat[flat_idx] = ypred
    Tw2d = Tw_flat.reshape((H, W))
    Tw_plot = np.where(pred_domain, Tw2d, np.nan).astype(np.float32)

    return {
        "time": t0,
        "Tw_plot": Tw_plot,
        "n_target": int(n),
        "valid_scene_frac": vfrac_scene,
        "filled_stream": int(flag_stream.sum()),
        "filled_temp": int(flag_temp.sum())
    }


# =============================================================================
# SINGLE-SCENE WORKFLOW
# =============================================================================
def run_single_scene(ctx):
    log("\n[RUN] SINGLE-SCENE PREDICTION")

    ti0 = pick_time_index(ctx["ds"], ctx["times"], ctx["stack_path"])
    res = predict_scene(ctx, ti0)

    t0 = pd.Timestamp(res["time"])
    Tw_plot = res["Tw_plot"]

    tif_path = OUTDIR / "Tw_pred_one_time.tif"
    save_float32_geotiff(tif_path, Tw_plot, ctx["transform"], crs="EPSG:4326")
    ok(f"Saved single-scene GeoTIFF: {tif_path}")

    title = f"Predicted river water temperature — {t0.strftime('%d %b %Y, %H:%M UTC')}"
    png_proj = OUTDIR / "Tw_pred_one_time_PROJECTED_PUBLICATION.png"
    vmin, vmax = make_single_map(
        arr2d=Tw_plot,
        transform=ctx["transform"],
        basin_gdf=ctx["basin"],
        river_gdf=ctx["river_in_basin"],
        out_png=png_proj,
        title=title,
        dst_crs=PLOT_CRS_PROJECTED,
        cmap=CMAP_TEMP,
        plot_river_underlay=PLOT_RIVER_UNDERLAY
    )
    ok(f"Saved projected single-scene figure: {png_proj}")

    png_wgs84 = OUTDIR / "Tw_pred_one_time_WGS84_PUBLICATION.png"
    make_single_map(
        arr2d=Tw_plot,
        transform=ctx["transform"],
        basin_gdf=ctx["basin"],
        river_gdf=ctx["river_in_basin"],
        out_png=png_wgs84,
        title=title,
        dst_crs=PLOT_CRS_WGS84,
        cmap=CMAP_TEMP,
        vmin=vmin, vmax=vmax,
        plot_river_underlay=PLOT_RIVER_UNDERLAY
    )
    ok(f"Saved WGS84 single-scene figure: {png_wgs84}")

    qc_txt = OUTDIR / "qc_summary_single_scene.txt"
    with open(qc_txt, "w", encoding="utf-8") as f:
        f.write(f"Selected scene time: {t0}\n")
        f.write(f"Valid scene fraction before fill: {res['valid_scene_frac']}\n")
        f.write(f"Filled along-stream pixels: {res['filled_stream']}\n")
        f.write(f"Filled temporal pixels: {res['filled_temp']}\n")
        f.write(f"Target predicted pixels: {res['n_target']}\n")
        f.write(f"Projected figure color range: {vmin:.3f} to {vmax:.3f} °C\n")
    ok(f"Saved single-scene QC summary: {qc_txt}")


# =============================================================================
# CLIMATOLOGY WORKFLOW
# =============================================================================
def run_climatology(ctx):
    log("\n[RUN] LONG-TERM CLIMATOLOGY")

    nt = len(ctx["times"])
    if MAX_SCENES_FOR_CLIMATOLOGY is None:
        scene_indices = list(range(nt))
    else:
        scene_indices = list(range(min(MAX_SCENES_FOR_CLIMATOLOGY, nt)))

    H, W = ctx["H"], ctx["W"]

    sum_annual = np.zeros((H, W), dtype=np.float64)
    cnt_annual = np.zeros((H, W), dtype=np.uint32)

    season_sums = {s: np.zeros((H, W), dtype=np.float64) for s in ["DJF", "MAM", "JJA", "SON"]}
    season_cnts = {s: np.zeros((H, W), dtype=np.uint32) for s in ["DJF", "MAM", "JJA", "SON"]}

    n_used = 0
    season_scene_counts = {"DJF": 0, "MAM": 0, "JJA": 0, "SON": 0}
    n_skipped_read = 0

    for k, ti in enumerate(scene_indices, start=1):
        try:
            raw_lst = read_ecostress_scene_safe(ctx["ds"], ti, stack_path=ctx["stack_path"], var=LST_VAR)
        except Exception as e:
            warn(f"Skipping unreadable scene {ti}: {e}")
            n_skipped_read += 1
            continue

        raw_valid_frac = float(np.isfinite(raw_lst).mean())
        if raw_valid_frac < MIN_SCENE_VALID_FRAC_FOR_CLIM:
            continue

        res = predict_scene(ctx, ti)
        Tw = res["Tw_plot"]
        t0 = pd.Timestamp(res["time"])

        valid = np.isfinite(Tw)
        if valid.sum() == 0:
            continue

        sum_annual[valid] += Tw[valid]
        cnt_annual[valid] += 1

        season = season_from_month(t0.month)
        season_sums[season][valid] += Tw[valid]
        season_cnts[season][valid] += 1
        season_scene_counts[season] += 1
        n_used += 1

        if k % 50 == 0 or k == len(scene_indices):
            ok(f"Processed {k}/{len(scene_indices)} scenes | used={n_used} | skipped_read={n_skipped_read}")

    annual_mean = np.full((H, W), np.nan, dtype=np.float32)
    m = cnt_annual > 0
    annual_mean[m] = (sum_annual[m] / cnt_annual[m]).astype(np.float32)

    seasonal_means = {}
    for s in ["DJF", "MAM", "JJA", "SON"]:
        arr = np.full((H, W), np.nan, dtype=np.float32)
        ms = season_cnts[s] > 0
        arr[ms] = (season_sums[s][ms] / season_cnts[s][ms]).astype(np.float32)
        seasonal_means[s] = arr

    save_float32_geotiff(OUTDIR / "Tw_pred_mean_annual.tif", annual_mean, ctx["transform"], crs="EPSG:4326")
    for s in ["DJF", "MAM", "JJA", "SON"]:
        save_float32_geotiff(OUTDIR / f"Tw_pred_mean_{s}.tif", seasonal_means[s], ctx["transform"], crs="EPSG:4326")
    ok("Saved climatology GeoTIFFs.")

    maps_dict = {
        "Annual": annual_mean,
        "DJF": seasonal_means["DJF"],
        "MAM": seasonal_means["MAM"],
        "JJA": seasonal_means["JJA"],
        "SON": seasonal_means["SON"],
    }

    png_proj = OUTDIR / "Tw_pred_climatology_PROJECTED_PUBLICATION.png"
    vmin, vmax = make_climatology_panel(
        maps_dict=maps_dict,
        transform=ctx["transform"],
        basin_gdf=ctx["basin"],
        river_gdf=ctx["river_in_basin"],
        out_png=png_proj,
        title="Long-term mean predicted river water temperature",
        dst_crs=PLOT_CRS_PROJECTED,
        cmap=CMAP_TEMP,
        plot_river_underlay=PLOT_RIVER_UNDERLAY
    )
    ok(f"Saved projected climatology panel: {png_proj}")

    png_wgs84 = OUTDIR / "Tw_pred_climatology_WGS84_PUBLICATION.png"
    make_climatology_panel(
        maps_dict=maps_dict,
        transform=ctx["transform"],
        basin_gdf=ctx["basin"],
        river_gdf=ctx["river_in_basin"],
        out_png=png_wgs84,
        title="Long-term mean predicted river water temperature",
        dst_crs=PLOT_CRS_WGS84,
        cmap=CMAP_TEMP,
        plot_river_underlay=PLOT_RIVER_UNDERLAY
    )
    ok(f"Saved WGS84 climatology panel: {png_wgs84}")

    qc_txt = OUTDIR / "qc_summary_climatology.txt"
    with open(qc_txt, "w", encoding="utf-8") as f:
        f.write("Long-term climatology summary\n")
        f.write("----------------------------\n")
        f.write(f"Total candidate scenes: {len(scene_indices)}\n")
        f.write(f"Used scenes: {n_used}\n")
        f.write(f"Unreadable skipped scenes: {n_skipped_read}\n")
        f.write(f"DJF scene count: {season_scene_counts['DJF']}\n")
        f.write(f"MAM scene count: {season_scene_counts['MAM']}\n")
        f.write(f"JJA scene count: {season_scene_counts['JJA']}\n")
        f.write(f"SON scene count: {season_scene_counts['SON']}\n")
        f.write(f"Common color range for climatology panel: {vmin:.3f} to {vmax:.3f} °C\n")
        f.write(f"Annual valid pixels: {(cnt_annual > 0).sum()}\n")
        for s in ["DJF", "MAM", "JJA", "SON"]:
            f.write(f"{s} valid pixels: {(season_cnts[s] > 0).sum()}\n")
    ok(f"Saved climatology QC summary: {qc_txt}")


# =============================================================================
# MAIN
# =============================================================================
def main():
    log("\n=== PREPARING CONTEXT ===")
    ctx = prepare_context()

    if RUN_SINGLE_SCENE:
        run_single_scene(ctx)

    if RUN_CLIMATOLOGY:
        run_climatology(ctx)

    log("\n🎉 DONE.")


if __name__ == "__main__":
    main()

In [ ]:
# -*- coding: utf-8 -*-
"""
RF station-wise time series comparison (Obs vs RF CV prediction)

- Rebuilds df from MASTER_CSV (same filtering as training)
- Generates out-of-fold (5-fold CV) predictions for Random Forest
- Saves station-wise metrics
- Creates composite figure: 3 stacked time-series (left) + Map image (right)
- Shared legend placed above figure on top-right
- Metrics boxes placed just above each subplot (outside plotting area)
- Removes 'Water_' / 'Water ' from station names in titles (display only)
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from PIL import Image


# ============================================================
# USER SETTINGS
# ============================================================
MASTER_CSV = Path(r"C:\goyal_shekhar\post_doc\ecostress\ecostress_master_training_clean.csv")
OUT_DIR    = Path(r"C:\goyal_shekhar\post_doc\ecostress\figure_models\RF_timeseries")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAP_PNG = Path(r"C:\goyal_shekhar\post_doc\ecostress\predictions_rf_full_pubready_projected\Tw_pred_one_time_WGS84_PUBLICATION.png")

stations_3 = [
    "Water_Mahndorf",
    "Water_Silberhuette",
    "Water_Meisdorf",
]

MIN_POINTS_STATIONWISE = 25
N_SPLITS = 5
RANDOM_STATE = 42


# ============================================================
# HELPERS
# ============================================================
def safe_name(s):
    return "".join(ch if (ch.isalnum() or ch in ("_", "-", ".")) else "_" for ch in str(s))

def clean_station_label(st):
    st = str(st).strip()
    st = st.replace("Water_", "").replace("Water ", "")
    return st

def station_metric_text(g, y_col="T_w_obs", pred_col="RF_CV_pred"):
    yt = g[y_col].values
    yp = g[pred_col].values
    r2 = r2_score(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae = mean_absolute_error(yt, yp)
    return f"R² = {r2:.2f}\nRMSE = {rmse:.2f} °C\nMAE = {mae:.2f} °C"


# ============================================================
# GLOBAL PLOT STYLE
# ============================================================
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.titleweight": "bold",
    "axes.labelsize": 14,
    "axes.labelweight": "bold",
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "axes.linewidth": 1.1,
    "xtick.major.width": 1.1,
    "ytick.major.width": 1.1,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


# ============================================================
# FEATURES (must match training)
# ============================================================
buffer_cols = [
    "buf50m_mean", "buf100m_mean", "buf150m_mean",
    "buf200m_mean", "buf300m_mean", "buf400m_mean", "buf500m_mean",
]

feature_cols = [
    "center_mean", *buffer_cols,
    "Q", "T_air",
    "DEM_m", "slope_deg_or_pct",
    "lulc_10", "lulc_30", "lulc_40", "lulc_50", "lulc_60",
    "doy_sin", "doy_cos", "hour",
    "Q_month_mean", "Q_month_max",
    "T_month_mean", "T_month_max",
    "Q_season_mean", "Q_season_max",
    "T_season_mean", "T_season_max",
]

target_col = "T_w_obs"


# ============================================================
# LOAD + CLEAN
# ============================================================
print(f"[CHECK] MASTER_CSV: {MASTER_CSV}")
if not MASTER_CSV.exists():
    raise FileNotFoundError(f"MASTER_CSV not found: {MASTER_CSV}")
print("✅ MASTER_CSV exists.")
print(f"✅ OUT_DIR: {OUT_DIR}")

df = pd.read_csv(MASTER_CSV, parse_dates=["time"])
print(f"✅ Loaded raw df: {df.shape}")

for r in [50, 100, 150]:
    col = f"buf{r}m_mean"
    if col in df.columns:
        df[col] = df[col].fillna(df["center_mean"])

missing = [c for c in (feature_cols + ["station_name", "time", target_col]) if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing required columns in CSV: {missing}")

mask_valid = (~df[feature_cols].isna().any(axis=1)) & (~df[target_col].isna())
df = df.loc[mask_valid].reset_index(drop=True)
print(f"✅ Rows kept: {df.shape[0]}")

X = df[feature_cols].values
y = df[target_col].values


# ============================================================
# RF MODEL + OOF CV PREDICTIONS
# ============================================================
rf_base = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
y_pred_oof = np.full_like(y, np.nan, dtype=float)

for fold, (tr, te) in enumerate(kf.split(X), start=1):
    m = clone(rf_base)
    m.fit(X[tr], y[tr])
    y_pred_oof[te] = m.predict(X[te])
    print(f"Fold {fold} done.")

df_eval = df[["time", "station_name", target_col]].copy()
df_eval["RF_CV_pred"] = y_pred_oof


# ============================================================
# Station-wise metrics CSV
# ============================================================
rows = []
for st, g in df_eval.groupby("station_name"):
    if len(g) < MIN_POINTS_STATIONWISE:
        continue

    yt = g[target_col].values
    yp = g["RF_CV_pred"].values

    rows.append({
        "station_name": st,
        "station_label": clean_station_label(st),
        "n": len(g),
        "R2": r2_score(yt, yp),
        "RMSE": np.sqrt(mean_squared_error(yt, yp)),
        "MAE": mean_absolute_error(yt, yp),
    })

st_metrics = pd.DataFrame(rows).sort_values("R2", ascending=False)
st_metrics_path = OUT_DIR / "RF_station_metrics_OOF.csv"
st_metrics.to_csv(st_metrics_path, index=False)
print(f"✅ Saved: {st_metrics_path}")


# ============================================================
# COMPOSITE FIGURE
# ============================================================
print("\n[STEP 8] Composite figure (3 stacked TS + map on right)")

if not MAP_PNG.exists():
    raise FileNotFoundError(f"Map image not found: {MAP_PNG}")

FS_TITLE  = 16
FS_LABEL  = 14
FS_TICK   = 13
FS_LEG    = 13
FS_METRIC = 12

fig = plt.figure(figsize=(17.2, 8.3))
gs = GridSpec(
    nrows=3, ncols=2,
    width_ratios=[1.05, 2.35],
    height_ratios=[1, 1, 1],
    wspace=0.12,
    hspace=0.62,
    figure=fig
)

ax_ts = [fig.add_subplot(gs[i, 0]) for i in range(3)]
ax_map = fig.add_subplot(gs[:, 1])

legend_handles = None
legend_labels = None

for i, st in enumerate(stations_3):
    ax = ax_ts[i]
    g = df_eval[df_eval["station_name"] == st].sort_values("time")

    if g.empty:
        ax.axis("off")
        continue

    line_obs, = ax.plot(
        g["time"], g[target_col],
        lw=2.5,
        label="Observed",
        zorder=3
    )
    line_pred, = ax.plot(
        g["time"], g["RF_CV_pred"],
        lw=2.5,
        ls="--",
        label="RF prediction",
        zorder=3
    )

    if legend_handles is None:
        legend_handles = [line_obs, line_pred]
        legend_labels = ["Observed", "RF prediction"]

    ax.set_title(
        clean_station_label(st),
        fontsize=FS_TITLE,
        fontweight="bold",
        pad=8
    )

    ax.grid(True, alpha=0.18, linestyle="--")

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=FS_TICK,
        width=1.2,
        length=5
    )

    ax.xaxis.set_major_locator(mdates.YearLocator(1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.set_ylabel("Tw (°C)", fontsize=FS_LABEL, fontweight="bold")

    if i == 2:
        ax.set_xlabel("Year", fontsize=FS_LABEL, fontweight="bold")
    else:
        ax.set_xlabel("")

    ax.margins(y=0.12)

    metric_txt = station_metric_text(g, y_col=target_col, pred_col="RF_CV_pred")

    # Metrics box ABOVE axes so it does not intersect the data
    ax.annotate(
        metric_txt,
        xy=(0.0, 1.0),
        xycoords="axes fraction",
        xytext=(0, 10),
        textcoords="offset points",
        ha="left",
        va="bottom",
        fontsize=FS_METRIC,
        fontweight="bold",
        bbox=dict(
            facecolor="white",
            edgecolor="0.6",
            boxstyle="round,pad=0.25"
        ),
        annotation_clip=False
    )

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

# Shared legend on top-left, above plots
if legend_handles is not None:
    fig.legend(
        legend_handles,
        legend_labels,
        loc="upper left",
        bbox_to_anchor=(0.19, 1.01),
        ncol=2,
        frameon=False,
        fontsize=FS_LEG,
        handlelength=2.8,
        columnspacing=1.5
    )

# Map panel
im = Image.open(MAP_PNG)
ax_map.imshow(im)
ax_map.axis("off")

# Adjust layout so top legend and metric boxes have room
fig.subplots_adjust(
    left=0.055,
    right=0.99,
    top=0.90,
    bottom=0.07
)

# Save
out_png = OUT_DIR / "RF_composite_3TS_plus_Map_pubready.png"
out_pdf = OUT_DIR / "RF_composite_3TS_plus_Map_pubready.pdf"



plt.show()
plt.close(fig)

print("✅ Saved:", out_png)
print("✅ Saved:", out_pdf)
print("\n✅ DONE.")